# Conformal Risk Control for Startup Success Prediction

A classifier predicts whether a startup succeeds; Conformal Risk Control (CRC,
Angelopoulos et al., 2024) then picks the decision threshold that bounds the
**False Negative Rate** of the success class at a target level `alpha`. Missing a
successful startup is the costly error, so the FNR is the risk under control.

Three questions:

- **RQ1** — does a single, global CRC threshold keep the test FNR at `alpha`, and
  what does it cost on the other metrics?
- **RQ2** — with that global threshold, how is the FNR distributed across
  subgroups (industry sector, founding-year cohort)?
- **RQ3** — what changes when the threshold is calibrated per subgroup
  (group-level, or Mondrian, CRC)?

## Weights & Biases is optional

With `USE_WANDB = False` every model of the sweep grid in `config.yaml` is trained
locally and nothing is logged remotely; the results are kept in memory, which is
all the cells below need. With it on, a sweep agent drives the same runs and logs
them to the entity and project read from `.env`. It is the same `train` function
either way.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from itertools import product
from types import SimpleNamespace
from dotenv import load_dotenv 
import yaml

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

from src.encoding import apply_frequency_encoding, fit_frequency_encoding
from src.models import build
from src.utils import (set_seed,
                       run_conformal_fnr_experiment,
                       run_conformal_fnr_group_calibrated, plot_conformal_fnr,
                       assign_model_colors, fnr_by_group_table,
                       crc_summary_tables, plot_crc_metric_boxplots,
                       plot_fnr_by_group_boxplots, plot_fnr_by_group_lines)


from sklearn.metrics import accuracy_score, roc_auc_score
import wandb
import matplotlib.pyplot as plt 


import scipy.integrate
if not hasattr(np, 'trapz'):
    np.trapz = scipy.integrate.trapezoid

load_dotenv()

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)


timeWindow=int(config['time_window'])

lastYear=int(config['last_year'])

# True: one W&B sweep run per model (logs online). False: same grid, run locally,
# nothing logged to W&B (as in the upstream repo).
USE_WANDB = False

conformal_store = {}
conformal_pool_store = {}

---
## Splits, encoding, imputation and scaling

The dataset is split once, stratified on the target, into a training set, a
validation set (early stopping) and a **conformal pool**. The pool is never seen
during training: the conformal experiments below re-split it 50/50 into a
calibration half and a test half, once per seed.

The frequency encoding of the categorical columns is fitted on the training rows
alone, so no held-out row contributes to the encoding applied to it. The raw
sector names are kept aside for the subgroup analyses.

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset"])

X = dataset.drop(['CompanyID', 'Target'], axis=1)
y = dataset['Target']

train_frac = config['train_frac']
cal_frac   = config['calibration_frac']
val_frac   = config['validation_frac']
test_frac  = config['test_frac']
seed       = config['random_seed']

# Partial split (stratified): train / validation /  calibration & test = 50% / 10% / 20% + 20%.
# Final calibration-test split is done later in the multi-seed robustness check, which re-splits the calibration-test pool 50/50.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, train_size=train_frac, stratify=y, random_state=seed)

# val_frac is a share of the whole dataset, X_tmp holds (1 - train_frac) of it.
X_val, X_cal_test, y_val, y_cal_test = train_test_split(
    X_tmp, y_tmp, train_size=val_frac / (1 - train_frac), stratify=y_tmp, random_state=seed)


# Frequency encoding of the categorical columns, fitted on train only: a category
# share computed before the split would partly come from the rows it encodes.
# `dataset` keeps the raw names, used for the sector subgroups in RQ2/RQ3.
enc_cfg = config["frequency_encoding"]
X_train, X_val, X_cal_test = X_train.copy(), X_val.copy(), X_cal_test.copy()
for column in enc_cfg["columns"]:
    encoding = fit_frequency_encoding(X_train[column],
                                      min_frequency=enc_cfg["min_frequency"],
                                      other_label=enc_cfg["other_label"])
    for frame in (X_train, X_val, X_cal_test):
        frame[column] = apply_frequency_encoding(frame[column], encoding,
                                                 other_label=enc_cfg["other_label"])

# Imputation (fit on train, apply to all splits)
imputer = KNNImputer(n_neighbors=5)
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_cal_test_imp   = imputer.transform(X_cal_test)
    
# Scaling (fit on imputed train, apply to all splits)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_cal_test_scaled   = scaler.transform(X_cal_test_imp)

# Matrices in the layout the model wrappers read (Model.matrices in
# src/models/base.py): the "test" slot is the calibration+test pool.
split = {
    "X_train_imp": X_train_imp, "X_val_imp": X_val_imp, "X_test_imp": X_cal_test_imp,
    "X_train_scaled": X_train_scaled, "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_cal_test_scaled,
}



print(f"Total: {len(y)} | Train: {len(y_train)}| Val: {len(y_val)} | Cal+Test: {len(y_cal_test)}")

---
## Sweep

Only with `USE_WANDB` on: the sweep is created from `sweep_settings` in
`config.yaml`. Its grid fixes every hyperparameter, so the only axis it crosses is
the model family.

In [ ]:
if USE_WANDB:
    entity=os.getenv("entity")
    project=os.getenv("project")

    # Initialize a new sweep
    sweep_id = wandb.sweep(config["sweep_settings"],entity=entity, project=project)

---
## Training and conformal calibration

One run per model family. Each run trains the model, scores the conformal pool
once, and passes those probabilities to the conformal experiment: for every seed
in `conformal.seeds` the pool is re-split 50/50 into calibration and test, and for
every `alpha` in `conformal.alpha` the CRC threshold `t*` is fitted on the
calibration half and evaluated on the test half. Each seed also carries a *base*
row at the default threshold 0.5, as an uncalibrated reference.

Training accuracy and AUC are printed as an overfitting check. The pool
probabilities are stored per model, so the subgroup analyses below reuse them
without retraining.

In [ ]:
def train(wandb_config):

    # Reproducibility: seed Python, NumPy, PyTorch (CPU+CUDA), cuDNN.
    # The wrappers also pass the seed as random_state to every estimator.
    set_seed(config['random_seed'])
    
    # Each family picks its matrices (imputed, or imputed and scaled) and
    # its own fit; see src/models/.
    model = build(wandb_config.model_type, wandb_config, seed=config['random_seed'])
    X_tr, X_v, X_ct = model.matrices(split)
    model.fit(X_tr, y_train, X_v, y_val)

    probs_train, preds_train = model.score(X_tr)
    labels_train = y_train

    probs_cal_test = model.predict_proba(X_ct)
    labels_cal_test = y_cal_test

    # Training-set fit indicators (constant across seeds): overfitting check.
    acc_train = accuracy_score(labels_train, preds_train)
    try:
        auc_train = roc_auc_score(labels_train, probs_train)
    except ValueError:
        auc_train = float("nan")

    if USE_WANDB:
        wandb.log({"accuracy_train": acc_train, "AUC_train": auc_train})
    print(f"Train accuracy: {acc_train:.3f} | Train AUC: {auc_train:.3f}")

    # ── CONFORMAL RISK CONTROL ON FNR ──
    # Per-run metrics come from the conformal experiment on the cal+test
    # pool: for every seed a base row (no calibration, 0.5 threshold) and
    # one row per target alpha (conformal threshold t*). Metrics: AUC
    # (base only, threshold-free), accuracy, precision, recall, f1, FNR, FPR.
    conf_cfg = config.get("conformal", {})
    if conf_cfg.get("enabled", False):

        df_conf = run_conformal_fnr_experiment(
            probs_cal_test, labels_cal_test,
            alphas=conf_cfg["alpha"],
            seeds=conf_cfg["seeds"],
        )

        fig_conf = plot_conformal_fnr(df_conf, model_name=wandb_config.model_type)

        # Run-level scalars (mean over seeds): pre- (base, 0.5) and
        # post-calibration (t* per alpha).
        base = df_conf[~df_conf["calibrated"]]
        conf_log = {
            "conformal/AUC":            float(base["AUC"].mean()),
            "conformal/base/accuracy":  float(base["accuracy"].mean()),
            "conformal/base/precision": float(base["precision"].mean()),
            "conformal/base/recall":    float(base["recall"].mean()),
            "conformal/base/f1":        float(base["f1"].mean()),
            "conformal/base/FNR":       float(base["FNR"].mean()),
            "conformal/base/FPR":       float(base["FPR"].mean()),
        }
        for a in conf_cfg["alpha"]:
            sub = df_conf[df_conf["calibrated"] & (df_conf["alpha"] == a)]
            conf_log[f"conformal/conf/a{a}/accuracy"]  = float(sub["accuracy"].mean())
            conf_log[f"conformal/conf/a{a}/precision"] = float(sub["precision"].mean())
            conf_log[f"conformal/conf/a{a}/recall"]    = float(sub["recall"].mean())
            conf_log[f"conformal/conf/a{a}/f1"]        = float(sub["f1"].mean())
            conf_log[f"conformal/conf/a{a}/FNR"]       = float(sub["FNR"].mean())
            conf_log[f"conformal/conf/a{a}/FPR"]       = float(sub["FPR"].mean())
            conf_log[f"conformal/conf/a{a}/threshold"] = float(sub["threshold"].mean())

        if USE_WANDB:
            wandb.log({
                "conformal_fnr_table": wandb.Table(dataframe=df_conf),
                "conformal_fnr_plot": wandb.Image(fig_conf),
                **conf_log,
            })
        plt.close(fig_conf)

        mean_fnr = df_conf[df_conf["calibrated"]].groupby("alpha")["FNR"].mean()
        print("Conformal FNR control (mean realized FNR per alpha):")
        print(mean_fnr.to_string())

        conformal_store[wandb_config.model_type] = df_conf
        conformal_pool_store[wandb_config.model_type] = probs_cal_test

# Every combination the sweep grid declares (only model_type varies).
params = config["sweep_settings"]["parameters"]
runs = [dict(zip(params, combo)) for combo in product(*(p["values"] for p in params.values()))]

if USE_WANDB:
    def wandb_train():
        with wandb.init():
            train(wandb.config)

    wandb.agent(sweep_id, function=wandb_train, count=len(runs))
else:
    for run in runs:
        print(f"── {run['model_type']} ──")
        train(SimpleNamespace(**run))

---
## RQ1 — Global-level CRC

For each model, two tables (means over seeds): the metrics at the 0.5 threshold,
before calibration, and the metrics at the conformal threshold for each target
`alpha`. AUC does not depend on a threshold and is reported once.

In the plot, each point is the realized test FNR of one seed, the solid line their
mean and the band ± 1 standard deviation. The dashed line is `y = x`: the risk is
controlled when the mean lies on or below it. The CRC guarantee is
`E[FNR_test] <= alpha` over calibration/test splits, so single seeds can sit above
the line.

In [ ]:
for model_type, df_conf in conformal_store.items():
    print(f"=== {model_type.upper()} ===")
    base, conf = crc_summary_tables(df_conf)
    print("Pre-calibration metrics (mean over seeds):")
    display(base)
    print("Post-calibration metrics per target alpha (mean over seeds):")
    display(conf)
    plot_conformal_fnr(df_conf, model_name=model_type)
    plt.show()

### Metric distributions across seeds

One figure per metric, models side by side. Within each model the first (grey) box
is the 0.5 threshold, then one box per target `alpha`, from lightest (0.05) to
darkest (0.25). Each box is the distribution of the metric over the seeds.

Recall is `1 - FNR`, so it is pinned by `alpha`; the other metrics show the price
of that recall, mostly through the false positives a lower threshold lets in.

In [ ]:
# Global-level CRC: per-metric boxplots over seeds (one representative frame per
# model; the pool is fixed so every model's frame carries the same seeds).
model_df = {}
for model_type, df_conf in conformal_store.items():
    model_df.setdefault(model_type, df_conf)
plot_crc_metric_boxplots(model_df)

---
## RQ2 — Subgroups under global-level CRC

The CRC guarantee is **marginal**: it holds on the pool as a whole, not within each
subgroup. Here the single global threshold is kept, and its realized FNR is
measured within each industry sector and each founding-year cohort.

The pool labels and groups are computed once and shared by every model. A group
whose calibration half holds fewer than `min_group` positives is flagged
`infeasible`.

In [ ]:
# Group labels on the conformal pool (calibration + test rows, in pool order).
# Reused across models, since the pool is fixed.
pool_index   = X_cal_test.index
cohort_pool  = dataset.loc[pool_index, "YearFounded"].to_numpy()
labels_pool  = y_cal_test.to_numpy()

# Industry sector by name, read off the raw column (the features carry its
# train-fitted frequency encoding instead). The few rows without a sector form an
# "Unknown" group, flagged infeasible by min_group.
sector_pool = dataset.loc[pool_index, "PrimaryIndustrySector"].fillna("Unknown").to_numpy()

alphas = config["conformal"]["alpha"]
seeds  = config["conformal"]["seeds"]

model_color = assign_model_colors(list(conformal_pool_store))

# Compute the per-(model, dimension) tables once; keep the raw per-seed marginal
# AND conditional frames (the latter feeds RQ3). Sector first, then cohort.
subgroup_dims = [("Industry sector", sector_pool),
                 ("YearFounded cohort", cohort_pool)]
subgroup_tables = {}
subgroup_raw_marg = {}
subgroup_raw_cond = {}
for model_type, probs_pool in conformal_pool_store.items():
    for dim_name, groups in subgroup_dims:
        tab, raw_marg, raw_cond = fnr_by_group_table(
            probs_pool, labels_pool, groups, alphas, seeds, min_group=20)
        subgroup_tables[(model_type, dim_name)] = tab
        subgroup_raw_marg[(model_type, dim_name)] = raw_marg
        subgroup_raw_cond[(model_type, dim_name)] = raw_cond

### Industry sector

`FNR_marg` is the mean realized FNR of the group over the seeds; `FNR_marg - alpha`
is the gap above the target, blank when the group is at or below it. `n_pool` and
`n_pos_pool` are the size and the number of positives of the group in the whole
pool (calibration and test halves together); a group with no positives has an
undefined FNR, shown as `NaN`.

In the boxplots, one figure per `alpha`: groups on the x axis, one box per model,
each box the distribution of the realized FNR over the seeds. Boxes above the
dashed line are under-covered, boxes below it over-covered. Small groups have few
positives per test half, so expect taller boxes there.

In [ ]:
# Industry sector — table then grouped boxplot (models side by side, one figure
# per alpha).
marg_cols = ["group", "alpha", "FNR_marg", "FNR_marg - alpha",
             "n_pool", "n_pos_pool"]
dim_name = "Industry sector"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} ===")
    display(tab[marg_cols].rename(columns={"group": dim_name}))

sector_raw = {m: raw for (m, d), raw in subgroup_raw_marg.items() if d == dim_name}
plot_fnr_by_group_boxplots(sector_raw, dim_name, model_color,
                           ylabel="realized FNR (marginal)")

### Founding-year cohort

Same table and boxplots, with the cohorts (`YearFounded`) as groups. The line plot
adds a per-model view across the cohorts: mean realized FNR over the seeds,
± 1 standard deviation band, and the target `alpha` as a dashed line.

In [ ]:
# YearFounded cohort — table, grouped boxplot, then per-model line plot.
marg_cols = ["group", "alpha", "FNR_marg", "FNR_marg - alpha",
             "n_pool", "n_pos_pool"]
dim_name = "YearFounded cohort"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} ===")
    display(tab[marg_cols].rename(columns={"group": dim_name}))

cohort_raw = {m: raw for (m, d), raw in subgroup_raw_marg.items() if d == dim_name}
plot_fnr_by_group_boxplots(cohort_raw, dim_name, model_color,
                           ylabel="realized FNR (marginal)")
plot_fnr_by_group_lines(cohort_raw, dim_name, model_color,
                        ylabel="realized FNR (marginal)")

---
## RQ3 — Group-level CRC

Group-level (Mondrian) CRC calibrates a **separate threshold per subgroup**, on
that group's calibration rows, and applies it to the same group in the test half.
The guarantee then holds within each group instead of only on average. Both
groupings of RQ2 are used, sector and cohort.

An `infeasible` group does not have enough calibration positives for a per-group
threshold to mean anything; it falls back to the global threshold and stays
flagged in the tables.

### Global metrics under group-level calibration

The per-group thresholds are applied, and the metrics are measured on the whole
test half, as in RQ1, so the tables and boxplots read the same way. The
`threshold` column is empty: there is no single threshold any more.

In [ ]:
# Group-calibrated global CRC per (model, grouping). Reuses the same pool.
group_cal_store = {}
for model_type, probs_pool in conformal_pool_store.items():
    for dim_name, groups in subgroup_dims:
        group_cal_store[(model_type, dim_name)] = run_conformal_fnr_group_calibrated(
            probs_pool, labels_pool, groups, alphas, seeds, min_group=20)

In [ ]:
# Summary tables (pre/post) per model, one block per grouping.
for dim_name, _ in subgroup_dims:
    print(f"################  Group-level CRC calibrated by {dim_name}  ################")
    for (model_type, d), df_gc in group_cal_store.items():
        if d != dim_name:
            continue
        print(f"=== {model_type.upper()} — calibrated by {dim_name} ===")
        base, conf = crc_summary_tables(df_gc)
        print("Pre-calibration metrics (mean over seeds):")
        display(base)
        print("Post-calibration metrics per target alpha (mean over seeds):")
        display(conf)

In [ ]:
# Per-metric global boxplots, one set of figures per grouping.
for dim_name, _ in subgroup_dims:
    print(f"################  Group-level CRC calibrated by {dim_name}  ################")
    model_df = {m: df for (m, d), df in group_cal_store.items() if d == dim_name}
    plot_crc_metric_boxplots(model_df,
                             title_prefix=f"Group-level CRC (by {dim_name})")

### Mondrian minus standard CRC

The difference, model by model and `alpha` by `alpha`, between the metrics of the
group-level version and those of the global one of RQ1: Δ = Mondrian − standard,
means over seeds. Both versions run on the same pool and the same seeds, so the
rows are paired.

For FNR and FPR a positive Δ means more errors under Mondrian; for accuracy,
precision, recall and F1 it means Mondrian does better. Both versions target the
same `alpha`, so ΔFNR should stay close to zero: what remains is the cost, on the
pooled metrics, of equalizing the FNR across groups.

In [ ]:
# Metric deltas: Mondrian (group-level, RQ3) − CRC standard (marginal, RQ1).
# Standard  -> conformal_store[model]                 (single global threshold)
# Mondrian  -> group_cal_store[(model, grouping)]     (per-group thresholds)
# Same fixed pool and same seeds, so the two are directly comparable row by row.
metric_cols = ["accuracy", "precision", "recall", "f1", "FNR", "FPR"]

def _conf_means(df):
    """Mean-over-seeds conformal metrics indexed by target alpha."""
    return df[df["calibrated"]].groupby("alpha")[metric_cols].mean()

crc_mondrian_delta = {}
for dim_name, _ in subgroup_dims:
    blocks = []
    for model_type in conformal_store:
        std  = _conf_means(conformal_store[model_type])
        mond = _conf_means(group_cal_store[(model_type, dim_name)])
        delta = (mond - std)
        delta.insert(0, "model", model_type.upper())
        blocks.append(delta.reset_index())
    tab = (pd.concat(blocks, ignore_index=True)[["model", "alpha"] + metric_cols]
             .set_index(["model", "alpha"])
             .round(4))
    crc_mondrian_delta[dim_name] = tab
    print(f"################  Δ (Mondrian − standard CRC) — grouping: {dim_name}  ################")
    display(tab)

### Subgroups under group-level calibration

The RQ2 tables and plots again, now with the per-group threshold. `FNR_cond` is
the realized FNR of each group under its own threshold, next to `FNR_marg` from
RQ2 as a reference; `FNR_cond - alpha` is the remaining gap. Rows flagged
`infeasible` used the global threshold, so for them `FNR_cond` equals `FNR_marg`
and carries no group-level guarantee.

Industry sector first, then founding-year cohort with its line plot.

In [ ]:
# Industry sector — conditional table then grouped boxplot.
cond_cols = ["group", "alpha", "FNR_marg", "FNR_cond", "FNR_cond - alpha",
             "n_pool", "n_pos_pool", "infeasible"]
dim_name = "Industry sector"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} (group-calibrated) ===")
    display(tab[cond_cols].rename(columns={"group": dim_name}))

sector_cond = {m: raw for (m, d), raw in subgroup_raw_cond.items() if d == dim_name}
plot_fnr_by_group_boxplots(sector_cond, dim_name, model_color,
                           ylabel="realized FNR (group-calibrated)")

In [ ]:
# YearFounded cohort — conditional table, grouped boxplot, per-model line plot.
cond_cols = ["group", "alpha", "FNR_marg", "FNR_cond", "FNR_cond - alpha",
             "n_pool", "n_pos_pool", "infeasible"]
dim_name = "YearFounded cohort"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} (group-calibrated) ===")
    display(tab[cond_cols].rename(columns={"group": dim_name}))

cohort_cond = {m: raw for (m, d), raw in subgroup_raw_cond.items() if d == dim_name}
plot_fnr_by_group_boxplots(cohort_cond, dim_name, model_color,
                           ylabel="realized FNR (group-calibrated)")
plot_fnr_by_group_lines(cohort_cond, dim_name, model_color,
                        ylabel="realized FNR (group-calibrated)")